# Task description
- Translate text from Chinese to English.
- Main goal: Get familiar with transformer.

## install the required package

In [1]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/DL_Lab3/"

In [2]:
!pip install torchmetrics

## Import package

In [3]:
import os
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary

## Fix random seed

In [4]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(87)

# Data
- Original dataset is [20k-en-zh-translation-pinyin-hsk](https://huggingface.co/datasets/swaption2009/20k-en-zh-translation-pinyin-hsk)
- We select 50000 English-Chinese sentence pairs for translation task

- Args:
  - BATCH_SIZE
  - data_dir: the path to the given translation dataset
- Tokenizer: BertTokenizer
  - encode: convert text to token ID
  - decode: convert token ID back to text
- Add paddings
  - make all the sentences the same length by inserting token ID = PAD_IDX at the back

In [5]:
data_dir = "./translation_data.json"
BATCH_SIZE = 64

## Show the raw data

In [6]:
translation_raw_data = pd.read_json(data_dir)
translation_raw_data = translation_raw_data
display(translation_raw_data)

,english,chinese
0,"Slowly and not without struggle, America began...",美国缓慢地开始倾听，但并非没有艰难曲折。
1,Dithering is a technique that blends your colo...,抖动是关于颜色混合的技术，使你的作品看起来更圆滑，或者只是创作有趣的材质。
2,This paper discusses the petrologic characteri...,本文以珲春早第三纪含煤盆地的地质构违背景为依据，分析了煤系地层的岩石学特征。
3,The second encounter relates to my grandfather...,第二次事件跟我爷爷的宝贝匣子有关。
4,One way to address these challenges would be t...,解决这些挑战的途径包括依照麻瓜在南非的经验设立真相与和解委员会。
...,...,...
49995,You were too obtuse to take the hint.,你太迟钝了， 没有理解这种暗示。
49996,"Therefore, in the event the mortgagee of ship ...",因此，在这种情况下船舶抵押权人放弃了债务人提供的担保就会影响其他担保人的利益，导致抵押权人的...
49997,"Fourth, puncture administrative bloat.",第四，削弱行政膨胀。
49998,Massimo Oddo says he won't be thinking about h...,马西莫。奥多声明他不会在世界杯决赛圈比赛结束之前考虑未来的俱乐部。


## Tokenizer

In [7]:
from transformers import BertTokenizer
tokenizer_en = BertTokenizer.from_pretrained("bert-base-cased")
tokenizer_cn = BertTokenizer.from_pretrained("bert-base-chinese")

In [8]:
english_seqs = translation_raw_data["english"].apply(lambda x: tokenizer_en.encode(x, add_special_tokens=True, padding=False))
chinese_seqs = translation_raw_data["chinese"].apply(lambda x: tokenizer_cn.encode(x, add_special_tokens=True, padding=False))

MAX_TOKENIZE_LENGTH = max(english_seqs.str.len().max(),chinese_seqs.str.len().max()) # longest string
MAX_TOKENIZE_LENGTH = pow(2, math.ceil(math.log(MAX_TOKENIZE_LENGTH)/math.log(2)))   # closest upper to the power of 2

print("max tokenize length:", MAX_TOKENIZE_LENGTH)

max tokenize length: 128


## Add paddings

In [9]:
PAD_IDX = 0
BOS_IDX = chinese_seqs.iloc[0][0]
EOS_IDX = chinese_seqs.iloc[0][-1]

def add_padding(token_list, max_length):
    ### TO-DO: Add padding to make all the sentence the same length
    padded_list = token_list.copy()
    while len(padded_list) < max_length:
        padded_list.append(PAD_IDX)
    return padded_list

chinese_seqs = chinese_seqs.apply(lambda x: add_padding(x,MAX_TOKENIZE_LENGTH))
english_seqs = english_seqs.apply(lambda x: add_padding(x,MAX_TOKENIZE_LENGTH))

In [10]:
# check the padding result
print("=====chinese tokenized data=====")
print(chinese_seqs.iloc[0])

print("=====english tokenized data=====")
print(english_seqs.iloc[0])

=====chinese tokenized data=====
[101, 5401, 1744, 5353, 2714, 1765, 2458, 1993, 967, 1420, 8024, 852, 2400, 7478, 3766, 3300, 5680, 7410, 3289, 2835, 511, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
=====english tokenized data=====
[101, 13060, 1105, 1136, 1443, 5637, 117, 1738, 1310, 1106, 5113, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Datalodader
- Split dataset into training dataset(90%) and validation dataset(10%).
- Create dataloader to iterate the data.

In [11]:
data_size = len(translation_raw_data)
train_size = int(0.9*data_size)
test_size = data_size - train_size
print("train_size:",train_size)
print("test_size:",test_size)

en_training_data = []
cn_training_data = []
en_testing_data = []
cn_testing_data = []

for i in range(data_size):
    if (i < train_size):
        en_training_data.append(torch.Tensor(english_seqs.iloc[i]))
        cn_training_data.append(torch.Tensor(chinese_seqs.iloc[i]))
    else:
        en_testing_data.append(torch.Tensor(english_seqs.iloc[i]))
        cn_testing_data.append(torch.Tensor(chinese_seqs.iloc[i]))


class TextTranslationDataset(Dataset):
    def __init__(self, src, dst):
        self.src_list = src
        self.dst_list = dst

    def __len__(self):
        return len(self.src_list)

    def __getitem__(self, idx):
        return self.src_list[idx], self.dst_list[idx]

cn_to_en_train_set = TextTranslationDataset(cn_training_data, en_training_data)
cn_to_en_test_set = TextTranslationDataset(cn_testing_data, en_testing_data)

cn_to_en_train_loader = DataLoader(cn_to_en_train_set, batch_size=BATCH_SIZE, shuffle=False)
cn_to_en_test_loader = DataLoader(cn_to_en_test_set, batch_size=BATCH_SIZE, shuffle=True)

train_size: 45000
test_size: 5000


# Model
- TO-DO: Finish the model by yourself
- Base transformer layers in [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
    - TransformerEncoderLayer:
    - TransformerDecoderLayer:
- Positional encoding and input embedding
- Note that you may need masks when implementing attention mechanism
    - Padding mask: prevent input from attending to padding tokens
    - Causal mask: prevent decoder input from attending to future input

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, head_num):
        super().__init__()
        self.d_model = d_model
        self.head_num = head_num
        self.head_dim = d_model // head_num
        # 創建線性層來轉換輸入到查詢(Q)、鍵(K)和值(V)
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)

        self.out_linear = nn.Linear(d_model, d_model)
        self.layer_norm = nn.LayerNorm(d_model)

    def forward(self, Q, K, V, src_padding_mask=None, future_mask=None):
        q_seq_len, batch_size, _ = Q.shape# 獲取輸入的維度信息
        k_seq_len = K.shape[0]
        enc_inp = Q# 保存輸入用於殘差連接
        Q = self.q_linear(Q) # 通過線性層轉換輸入
        K = self.k_linear(K)
        V = self.v_linear(V)
        # 重塑並轉置張量以準備多頭注意力計算、 最終形狀: (batch_size, head_num, seq_len, head_dim)
        Q = Q.view(q_seq_len, batch_size, self.head_num, self.head_dim).transpose(0, 1).transpose(1, 2)
        K = K.view(k_seq_len, batch_size, self.head_num, self.head_dim).transpose(0, 1).transpose(1, 2)
        V = V.view(k_seq_len, batch_size, self.head_num, self.head_dim).transpose(0, 1).transpose(1, 2)
        # 計算注意力分數
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # 應用源序列的填充遮罩
        if src_padding_mask is not None:
            src_padding_mask = src_padding_mask.unsqueeze(1).unsqueeze(2).expand(-1, self.head_num, q_seq_len, -1)# 擴展遮罩以匹配注意力分數的形
            scores = scores.masked_fill(src_padding_mask, float('-inf'))# 將遮罩位置的分數設為負無窮大

        if future_mask is not None:
            future_mask = future_mask.unsqueeze(0).unsqueeze(1).expand(batch_size, self.head_num, -1, -1)# 擴展遮罩以匹配注意力分數的形狀
            future_mask = future_mask.to(torch.bool)  # 确保future_mask是布尔类型
            scores = scores.masked_fill(future_mask, float('-inf'))# 將遮罩位置的分數設為負無窮大
        # 應用softmax來獲得注意力權重
        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)# 將注意力權重應用到值(V)上
        output = output.transpose(1, 2).contiguous().view(batch_size, q_seq_len, self.d_model).transpose(0, 1)# 重塑輸出: (seq_len, batch_size, d_model)
        output = self.out_linear(output)# 通過輸出線性層
        output = self.layer_norm(output)# 應用層歸一化
        return output + enc_inp# 添加殘差連接並返回

In [13]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, dim_feedforward, nhead, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead)
        self.feed_forward = nn.Sequential(# 創建前饋神經網絡
            nn.Linear(d_model, dim_feedforward),# 第一個線性層，將維度從 d_model 擴展到 dim_feedforward
            nn.GELU(),# GELU 激活函數，相比 ReLU 有更好的性能
            nn.Linear(dim_feedforward, d_model)# 第二個線性層，將維度從 dim_feedforward 縮小回 d_model
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_padding_mask):
        attn_output = self.self_attn(x, x, x, src_padding_mask=src_padding_mask)# 將 x 作為查詢(Q)、鍵(K)和值(V)傳入自注意力層
        x = self.norm1(x + self.dropout(attn_output)) # 添加殘差連接（x +），應用 dropout，然後進行層歸一化
        ff_output = self.feed_forward(x)
        x = self.norm2(x)
        x = x + self.dropout(ff_output)

        return x

In [14]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, dim_feedforward, nhead, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead)# 創建多頭自注意力層，用於目標序列的自注意力
        self.cross_attn = MultiHeadAttention(d_model, nhead)# 創建多頭交叉注意力層，用於關注編碼器的輸出
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Linear(dim_feedforward, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)# 用於自注意力輸出後的歸一化
        self.norm2 = nn.LayerNorm(d_model)# 用於交叉注意力輸出後的歸一化
        self.norm3 = nn.LayerNorm(d_model)# 用於前饋網絡輸出後的歸一化
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_padding_mask=None, tgt_padding_mask=None, tgt_future_mask=None):
        attn_output = self.self_attn(x, x, x, src_padding_mask=tgt_padding_mask, future_mask=tgt_future_mask) # 自注意力層
        x = self.norm1(x + self.dropout(attn_output))

        attn_output = self.cross_attn(x, enc_output, enc_output, src_padding_mask=src_padding_mask) # 交叉注意力層
        x = self.norm2(x + self.dropout(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

In [15]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, num_encoder_layers, num_decoder_layers, d_ff, dropout):
        super().__init__()
        self.encoder_layers = nn.ModuleList([TransformerEncoderLayer(d_model, d_ff, num_heads, dropout) for _ in range(num_encoder_layers)])# 創建編碼器層列表
        self.decoder_layers = nn.ModuleList([TransformerDecoderLayer(d_model, d_ff, num_heads, dropout) for _ in range(num_decoder_layers)])# 創建解碼器層列表
        # 初始化權重
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src_embeded, tgt_embeded, src_padding_mask, tgt_padding_mask, tgt_future_mask):
        enc_output = self.encode(src_embeded, src_padding_mask)# 首先對源序列進行編碼
        dec_output = self.decode(tgt_embeded, enc_output, src_padding_mask, tgt_padding_mask, tgt_future_mask)# 然後使用編碼器輸出對目標序列進行解碼
        return dec_output

    def encode(self, src_embeded, src_padding_mask=None):# src_embeded: 源序列的嵌入表示 src_padding_mask: 源序列的填充遮罩
        for layer in self.encoder_layers:# 遍歷所有編碼器層
            src_embeded = layer(src_embeded, src_padding_mask)
        return src_embeded

    def decode(self, tgt_embeded, enc_output, src_padding_mask=None, tgt_padding_mask=None, tgt_future_mask=None):
        for layer in self.decoder_layers:
            tgt_embeded = layer(tgt_embeded, enc_output, src_padding_mask, tgt_padding_mask, tgt_future_mask)
        return tgt_embeded

In [16]:
class PositionalEncoding(nn.Module):
    def __init__(self, emb_size, dropout, maxlen=5000):
        super().__init__()
        den = torch.exp(- torch.arange(0, emb_size, 2) * math.log(10000) / emb_size)# 計算位置編碼的分母部分
        pos = torch.arange(0, maxlen).reshape(maxlen, 1)# 創建位置索引
        pos_embedding = torch.zeros((maxlen, emb_size)) #初始化位置嵌入矩陣
        pos_embedding[:, 0::2] = torch.sin(pos * den)# 使用正弦函數填充偶數索引列
        pos_embedding[:, 1::2] = torch.cos(pos * den)# 使用餘弦函數填充奇數索引列
        pos_embedding = pos_embedding.unsqueeze(-2) # 增加一個維度以便於後續的廣播操作

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('pos_embedding', pos_embedding)# 將位置嵌入註冊為緩衝區，這樣它就不會被視為模型參數

    def forward(self, token_embedding):
        return self.dropout(token_embedding + self.pos_embedding[:token_embedding.size(0), :])# 將位置編碼加到詞元嵌入上，然後應用dropout

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)# 將輸入轉換為長整型，進行嵌入查詢，然後乘以嵌入大小的平方根

In [17]:
def create_mask(src, tgt):
    src_seq_len, batch_size = src.shape
    tgt_seq_len = tgt.shape[0]

    src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)

    tgt_future_mask = torch.triu(torch.ones((tgt_seq_len, tgt_seq_len), device=DEVICE) == 1).transpose(0, 1)
    tgt_future_mask = tgt_future_mask.float().masked_fill(tgt_future_mask == 0, float('-inf')).masked_fill(tgt_future_mask == 1, float(0.0))

    return tgt_future_mask, src_padding_mask, tgt_padding_mask

In [18]:
# Seq2Seq Network
class Seq2SeqNetwork(nn.Module):
    def __init__(self,
                 num_encoder_layers,
                 num_decoder_layers,
                 emb_size,
                 nhead,
                 src_vocab_size,
                 tgt_vocab_size,
                 dim_feedforward,
                 dropout=0.1,
                 maxlen=5000):
        super().__init__()
        self.transformer = Transformer(
            d_model=emb_size,
            num_heads=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            d_ff=dim_feedforward,
            dropout=dropout
        )
        self.generator = nn.Linear(emb_size, tgt_vocab_size)
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout, maxlen=maxlen)

    def forward(self,
                src,
                trg,
                tgt_future_mask=None,
                src_padding_mask=None,
                tgt_padding_mask=None):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))
        outs = self.transformer(src_emb, tgt_emb, src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask, tgt_future_mask=tgt_future_mask)
        return self.generator(outs)


    def encode(self, src, src_padding_mask=None):
        return self.transformer.encode(self.positional_encoding(self.src_tok_emb(src)), src_padding_mask=src_padding_mask)

    def decode(self, tgt, memory, src_padding_mask=None, tgt_padding_mask=None, tgt_future_mask=None):
        return self.transformer.decode(self.positional_encoding(self.tgt_tok_emb(tgt)), memory, src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask, tgt_future_mask=tgt_future_mask)

## Note: The parameter size of model should be less than 100M (100,000k) !!!

In [19]:
EMB_SIZE = 512
NHEAD = 8
FFN_HID_DIM = 2048
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 6
SRC_VOCAB_SIZE = tokenizer_cn.vocab_size
TGT_VOCAB_SIZE = tokenizer_en.vocab_size
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

transformer = Seq2SeqNetwork(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE,
                                 NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM)

for p in transformer.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)

transformer = transformer.to(DEVICE)
param_transformer = sum(p.numel() for p in transformer.parameters())
print (f"The parameter size of transformer is {param_transformer/1000} k")
#   The parameter size of model should be less than 100M (100,000k) !!!
#   The parameter size of model should be less than 100M (100,000k) !!!
#   The parameter size of model should be less than 100M (100,000k) !!!

The parameter size of transformer is 84695.364 k


# Training
- You can change the training setting by yourself including
  - Number of epoch
  - Optimizer
  - Learning rate
  - Learning rate scheduler
  - etc...

In [20]:
NUM_EPOCHS = 30
# loss_fn = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX)
from torch.optim.lr_scheduler import ReduceLROnPlateau

# optimizer = torch.optim.Adam(transformer.parameters(), lr=0.00005, betas=(0.9, 0.98), eps=1e-9)
optimizer = torch.optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10, verbose=True)
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1, dim=-1, ignore_index=None):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes
        self.dim = dim
        self.ignore_index = ignore_index

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=self.dim)
        with torch.no_grad():
            # 确保 target 是正確的數據類型和形狀
            target = target.long().view(-1, 1)
            
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            if self.ignore_index is not None:
                true_dist[:, self.ignore_index] = 0
            true_dist.scatter_(1, target, self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=self.dim))

criterion = LabelSmoothingLoss(TGT_VOCAB_SIZE, smoothing=0.1, ignore_index=PAD_IDX)

c:\Users\User\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


## Translation quality metrics: BLEU score

In [21]:
from torchmetrics.text import BLEUScore

def bleu_score_func(predicted, truth, grams=1):
    preds = [predicted]
    truth = [[truth]]
    bleu = BLEUScore(n_gram=grams)
    return bleu(preds, truth)


def BLEU_batch(predict, truth, output_tokenizer):
    batch_size = predict.size(1)
    total_score = 0
    for i in range(batch_size):
        predict_str = output_tokenizer.decode(predict[:, i], skip_special_tokens=True)
        truth_str = output_tokenizer.decode(truth[:, i], skip_special_tokens=True)
        score_gram1 = bleu_score_func(predict_str.lower(), truth_str.lower(), grams=1)
        #score_gram2 = bleu_score_func(predict_str.lower(), truth_str, grams=2)
        #score_gram3 = bleu_score_func(predict_str.lower(), truth_str, grams=3)
        #score_gram4 = bleu_score_func(predict_str.lower(), truth_str, grams=4)
        #total_score = total_score + (score_gram1 + score_gram2 + score_gram3 + score_gram4) / 4.0
        total_score = total_score + score_gram1
    total_score = total_score / batch_size
    return total_score

## Training and Evaluation Functions

In [22]:
def train_epoch(model, optimizer, train_dataloader, criterion):
    model.train()
    losses = 0
    for i, (src, tgt) in enumerate(train_dataloader):
        src = src.transpose(0, 1).to(DEVICE)
        tgt = tgt.transpose(0, 1).to(DEVICE)

        tgt_input = tgt[:-1, :]
        tgt_future_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)

        logits = model(src, tgt_input, src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask, tgt_future_mask=tgt_future_mask)

        optimizer.zero_grad()

        tgt_out = tgt[1:, :]
        loss = criterion(logits.contiguous().view(-1, logits.size(-1)), tgt_out.contiguous().view(-1))
        loss.backward()

        # 添加梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        losses += loss.item()

        # if i % 100 == 0:
        #     print(f"Batch {i}, Loss: {loss.item():.4f}")

    return losses / len(train_dataloader)

def evaluate(model, val_dataloader, criterion):
    model.eval()
    losses = 0
    total_bleu = 0
    with torch.no_grad():
        for src, tgt in val_dataloader:
            src = src.transpose(0, 1).to(DEVICE)
            tgt = tgt.transpose(0, 1).to(DEVICE)

            tgt_input = tgt[:-1, :]
            tgt_future_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)

            logits = model(src, tgt_input, src_padding_mask=src_padding_mask, tgt_padding_mask=tgt_padding_mask, tgt_future_mask=tgt_future_mask)

            tgt_out = tgt[1:, :]
            loss = criterion(logits.contiguous().view(-1, logits.size(-1)), tgt_out.contiguous().view(-1))
            losses += loss.item()

            # 计算BLEU分数
            out_words = torch.max(logits, dim=-1)[1]
            bleu_score = BLEU_batch(out_words, tgt_out, tokenizer_en)
            total_bleu += bleu_score

    return losses / len(val_dataloader), total_bleu / len(val_dataloader)

## Start training
- MODEL_SAVE_PATH: path for storing the best model

In [23]:
MODEL_SAVE_PATH = "./model.ckpt"

In [24]:
from timeit import default_timer as timer
transformer = transformer.to(DEVICE)

best_val_loss = float('inf')

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = timer()
    train_loss = train_epoch(transformer, optimizer, cn_to_en_train_loader, criterion)
    end_time = timer()
    val_loss, val_bleu = evaluate(transformer, cn_to_en_test_loader, criterion)
    
    print(f"Epoch: {epoch}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}, Val BLEU: {val_bleu:.3f}, Epoch time = {(end_time - start_time):.3f}s")

    # 更新学习率
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(transformer.state_dict(), MODEL_SAVE_PATH)
        print("Model saved")


Epoch: 1, Train loss: 2.827, Val loss: 2.441, Val BLEU: 0.120, Epoch time = 333.984s
Model saved
Epoch: 2, Train loss: 2.404, Val loss: 2.388, Val BLEU: 0.156, Epoch time = 331.718s
Model saved
Epoch: 3, Train loss: 2.354, Val loss: 2.348, Val BLEU: 0.176, Epoch time = 331.799s
Model saved
Epoch: 4, Train loss: 2.315, Val loss: 2.319, Val BLEU: 0.191, Epoch time = 331.773s
Model saved
Epoch: 5, Train loss: 2.281, Val loss: 2.301, Val BLEU: 0.201, Epoch time = 331.445s
Model saved
Epoch: 6, Train loss: 2.252, Val loss: 2.280, Val BLEU: 0.207, Epoch time = 331.401s
Model saved
Epoch: 7, Train loss: 2.224, Val loss: 2.267, Val BLEU: 0.216, Epoch time = 331.636s
Model saved
Epoch: 8, Train loss: 2.197, Val loss: 2.251, Val BLEU: 0.225, Epoch time = 331.642s
Model saved
Epoch: 9, Train loss: 2.171, Val loss: 2.246, Val BLEU: 0.233, Epoch time = 331.747s
Model saved
Epoch: 10, Train loss: 2.146, Val loss: 2.235, Val BLEU: 0.239, Epoch time = 332.095s
Model saved
Epoch: 11, Train loss: 2.122,

# Inference

In [25]:
def generate_square_subsequent_mask(sz):
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-1e20')).masked_fill(mask == 1, float(0.0))
    return mask

# function to generate output sequence using greedy algorithm
def greedy_decode(model, src, src_mask, max_len, start_symbol):
    src = src.to(DEVICE)
    src_mask = src_mask.to(DEVICE)
    memory = model.encode(src, src_padding_mask=src_mask)
    ys = torch.zeros(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
    for i in range(max_len-1):
        memory = memory.to(DEVICE)
        tgt_padding_mask = torch.zeros(1, ys.size(0)).type(torch.bool).to(DEVICE)  # target padding mask
        tgt_mask = (generate_square_subsequent_mask(ys.size(0)).type(torch.bool)).to(DEVICE)  # target causal mask
        out = model.decode(ys, memory, src_padding_mask=src_mask, tgt_padding_mask=tgt_padding_mask, tgt_future_mask=tgt_mask)
        out = out.transpose(0, 1)
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat([ys, torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=0)
        if next_word == EOS_IDX:
            break
    return ys

# actual function to translate input sentence into target language
def translate(model: torch.nn.Module, src_sentence: str, input_tokenizer, output_tokenizer):
    model.eval()
    sentence = input_tokenizer.encode(src_sentence)
    sentence = torch.tensor(sentence).view(-1, 1)
    num_tokens = sentence.shape[0]

    src_mask = torch.zeros(1, num_tokens).type(torch.bool)  # source padding mask
    tgt_tokens = greedy_decode(model, sentence, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX).flatten()
    output_sentence = output_tokenizer.decode(tgt_tokens, skip_special_tokens=True)
    return output_sentence

## Load best model

In [26]:
transformer = Seq2SeqNetwork(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE,
                                 NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM)
transformer.to(DEVICE)
transformer.load_state_dict(torch.load("model.ckpt"))

C:\Users\User\AppData\Local\Temp\ipykernel_27564\2163959858.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  transformer.load_state_dict(torch.load("model.ckpt"))


<All keys matched successfully>

## Translation testing

In [27]:
sentence = "你好，欢迎来到中国"
ground_truth = 'Hello, Welcome to China'
predicted = translate(transformer, sentence, tokenizer_cn, tokenizer_en)

print(f'{"Input:":15s}: {sentence}')
print(f'{"Prediction":15s}: {predicted}')
print(f'{"Ground truth":15s}: {ground_truth}')
print("Bleu Score (1gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 1).item())
print("Bleu Score (2gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 2).item())
print("Bleu Score (3gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 3).item())
print("Bleu Score (4gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 4).item())

Input:         : 你好，欢迎来到中国
Prediction     : Hello, welcome China to go to China.
Ground truth   : Hello, Welcome to China
Bleu Score (1gram):  0.5714285969734192
Bleu Score (2gram):  0.30860671401023865
Bleu Score (3gram):  0.0
Bleu Score (4gram):  0.0


In [28]:
sentence = "早上好，很高心见到你"
ground_truth = 'Good Morning, nice to meet you'
predicted = translate(transformer, sentence, tokenizer_cn, tokenizer_en)

print(f'{"Input:":15s}: {sentence}')
print(f'{"Prediction":15s}: {predicted}')
print(f'{"Ground truth":15s}: {ground_truth}')
print("Bleu Score (1gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 1).item())
print("Bleu Score (2gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 2).item())
print("Bleu Score (3gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 3).item())
print("Bleu Score (4gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 4).item())

Input:         : 早上好，很高心见到你
Prediction     : Good morning, very happy, you're very happy.
Ground truth   : Good Morning, nice to meet you
Bleu Score (1gram):  0.2857142984867096
Bleu Score (2gram):  0.2182179093360901
Bleu Score (3gram):  0.0
Bleu Score (4gram):  0.0


In [29]:
sentence = "祝您有个美好的一天"
ground_truth = 'Have a nice day'
predicted = translate(transformer, sentence, tokenizer_cn, tokenizer_en)

print(f'{"Input:":15s}: {sentence}')
print(f'{"Prediction":15s}: {predicted}')
print(f'{"Ground truth":15s}: {ground_truth}')
print("Bleu Score (1gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 1).item())
print("Bleu Score (2gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 2).item())
print("Bleu Score (3gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 3).item())
print("Bleu Score (4gram): ", bleu_score_func(predicted.lower(), ground_truth.lower(), 4).item())

Input:         : 祝您有个美好的一天
Prediction     : I wish you have a good day.
Ground truth   : Have a nice day
Bleu Score (1gram):  0.2857142984867096
Bleu Score (2gram):  0.2182179093360901
Bleu Score (3gram):  0.0
Bleu Score (4gram):  0.0
